# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the [FAIR² Croissant dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL. All schema elements (record sets, fields, columns) are referenced via their `@id`.

In [ ]:
# Ensure mlcroissant is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # Note: Do not treat as dict

print(f"Dataset: {metadata.name}\nDescription: {metadata.description}\nVersion: {metadata.version}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

We examine all available record sets in the dataset and inspect fields and columns for one record set as an example.

In [ ]:
# List all record sets using their @id
print('Available record sets:')
record_sets = [rs for rs in metadata.record_sets]
for rs in record_sets:
    print(f"- @id: {rs['@id']} | name: {rs.get('name', '-')}")

# Let's choose the main protein summary table for inspection (assume @id from data or as seen below)
# If record_sets is empty (common in some schemas), we can try to read via dataset.record_sets
if len(record_sets) == 0:
    # fallback to dataset.record_sets (mlcroissant >=0.5.0)
    record_sets = dataset.record_sets
    print('\nRecord sets loaded with dataset.record_sets:')
    for rs in record_sets:
        print(f"- @id: {rs['@id']} | name: {rs.get('name', '-')}")
    # For most croissant-based datasets, first record_set is main table

# Select record set @id for further analysis:
record_set_id = record_sets[0]['@id']
print(f"\nSelected record set: {record_set_id}\n")

# List all fields for the selected record set
print("Fields (by @id and name):")
fields = record_sets[0].get('fields', [])

if len(fields) == 0:
    # fallback for classic Croissant
    fields = dataset.fields(record_set=record_set_id)

for fld in fields:
    print(f"- @id: {fld['@id']} | name: {fld.get('name', '-')}")

## 3. Data Extraction
Load data from the main record set into a pandas DataFrame for analysis. We use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set
dataframes = {}
rs_ids = [rs['@id'] for rs in record_sets]

for rsid in rs_ids:
    try:
        records = list(dataset.records(record_set=rsid))
        dataframes[rsid] = pd.DataFrame(records)
        print(f"Loaded record set: {rsid} with shape: {dataframes[rsid].shape}")
    except Exception as e:
        print(f"Error loading record set {rsid}: {e}")

# For demonstration, use the first record set
main_rs_id = rs_ids[0]
df = dataframes[main_rs_id]
print(f"\nColumns in main DataFrame (@id): {df.columns.tolist()}")
df.head()

## 4. Exploratory Data Analysis (EDA)
Let's apply data processing steps using field/column `@id`s.

- **Filtering:** We'll filter proteins with a minimum abundance threshold.
- **Normalization:** Abundance scores are normalized.
- **Grouping:** We'll aggregate by protein family if such a column exists.

In [ ]:
# Choose a numeric field; find candidate in list (example: 'abundance_sum' or similar)
print('Columns in DataFrame:')
for col in df.columns:
    print(f'- {col}')

# For demonstration, suppose '@id:protein_abundance' is the abundance field
# Replace with actual @id for abundance or relevant numeric value

numeric_field = None
for col in df.columns:
    if 'abundance' in col.lower() or 'mw' in col.lower() or 'coverage' in col.lower():
        numeric_field = col
        break

if numeric_field is None:
    numeric_field = df.select_dtypes(float).columns[0]
    print(f"No abundance/coverage column found; using first numeric: {numeric_field}")
else:
    print(f"Using numeric field: {numeric_field}")

threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 0
filtered_df = df[df[numeric_field] > threshold]
print(f"\nFiltered records with {numeric_field} > {threshold:.2f}:")
print(filtered_df.head())

# Normalization
filtered_df[numeric_field + '_normalized'] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()
) / filtered_df[numeric_field].std()
print(f"\nNormalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, numeric_field + '_normalized']].head())

# Try grouping by a categorical field, e.g. protein family, if present
group_field = None
for col in df.columns:
    if 'family' in col.lower() or 'category' in col.lower() or 'type' in col.lower():
        group_field = col
        break

if group_field and group_field in filtered_df:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
    print(f"\nGrouped data by {group_field} (mean {numeric_field}):")
    print(grouped_df.head())
else:
    print('\nNo suitable categorical field found for grouping.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using `matplotlib` or `seaborn`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the abundance/distribution
plt.figure(figsize=(8,5))
sns.histplot(df[numeric_field], kde=True)
plt.title(f'Distribution of {numeric_field}')
plt.xlabel(numeric_field)
plt.ylabel('Frequency')
plt.show()

# If there are two numeric columns, show scatter plot
numeric_cols = df.select_dtypes('number').columns
if len(numeric_cols) > 1:
    plt.figure(figsize=(7,5))
    sns.scatterplot(x=df[numeric_cols[0]], y=df[numeric_cols[1]])
    plt.xlabel(numeric_cols[0])
    plt.ylabel(numeric_cols[1])
    plt.title(f'Scatter plot: {numeric_cols[0]} vs {numeric_cols[1]}')
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated loading, inspecting, and processing a Croissant FAIR² dataset using `mlcroissant`. We explored the schema to identify record sets and fields via their `@id`, loaded a record set as a pandas DataFrame, filtered and normalized a key numeric feature, and produced basic visualizations. 

**Key insights**:
- The dataset provides structured mass spectrometry data suitable for downstream analysis.
- Field selection and transformations can be driven by `@id` to ensure reproducible and semantically robust analytics workflows.

You can further analyze and visualize specific proteins or sample conditions as required for your work.